# PONE-D-26-31931 — Reviewer 2, comment 5: CAT size-matched control

**Reviewer 2 asked:** is CAT's gain due to *compression diversity*, or simply to the ~5× larger training set? The test is to train on an equally sized dataset made only of CR1 (uncompressed) images.

**What this notebook trains (the "CR1×5 control"):** for every dataset and fold, the CAT run is repeated with **one change only**. The training data is the CR1 images repeated five times, instead of CR1 + CR15 + CR25 + CR50 + CR100. Everything else is copied from the notebooks that produced the reported CAT results:

| | BreakHis / BACH | NCT-CRC |
|---|---|---|
| Source notebook | `Breast Cancer Compression  Study` / `code on dataset 3 plos one` | `NCT-CRC JPEG2000 Compression Experiments` |
| Model | Inception-DenseNet169 + branch `ChannelAttention`, float32 | NCT concatenation + CBAM gate, **mixed float16** (as in its CAT run) |
| Test folds | the same patient-level 5-fold split as CAT (verified identical in Tier A) | the same fixed 1,170-image test set |
| Train/val split | 85/15 of training patients | the same `StratifiedKFold` indices as CAT, so each fold uses the same images |
| Training | Adam 1e-3, batch 32, ≤30 epochs, early stopping 5, reduce-LR 2, balanced class weights, the same augmentation | same, plus min_lr 1e-7 (as in its run) |

**Steps per epoch match CAT** because the training set has the same size, which is exactly what the reviewer asked to control.

**Analysis (Section 4)** splits CAT's benefit into two parts. The **volume effect** is control − robustness (more steps on CR1 only). The **diversity effect** is CAT − control (the compression variety itself). Both use the same paired bootstrap and Holm correction as Tier A, and need the Tier A `predictions/` folder.

**Runtime on L4:** about 2 h BreakHis + 10 min BACH + 3.5 h NCT-CRC. Each fold is saved as soon as it finishes. After a disconnect, re-run Sections 0–2 and the training cell; finished folds are skipped.

**Nothing existing is overwritten.** All outputs go to `R25_Control/` in your Drive.

## 0. Setup

In [1]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

import re, gc, glob, json, time, shutil, warnings
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import InceptionV3, DenseNet169
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from scipy import stats
warnings.filterwarnings('ignore')
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================
# CONFIG
# ============================================================
DRIVE = '/content/drive/MyDrive'
LOCAL = '/content/local_cr1' if IN_COLAB else '/tmp/local_cr1'   # fast local copy of CR1 training images

CFG = {
    'out_dir': f'{DRIVE}/R25_Control',
    'tierA_pred_dir': f'{DRIVE}/JPEG2000_Revision_TierA/predictions',
    'cr_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    'n_copies': 5,                      # CAT used 5 compression levels -> 5 copies of CR1
    'seed': 42, 'n_folds': 5, 'image_size': (224, 224), 'batch_size': 32,
    'epochs': 30, 'learning_rate': 0.001, 'early_stopping_patience': 5, 'reduce_lr_patience': 2,
    'n_boot': 10000,
    'datasets': {
        'BreakHis': {'loader': 'breakhis', 'data_dir': f'{DRIVE}/BreakHis_Split'},
        'BACH':     {'loader': 'bach',     'data_dir': f'{DRIVE}/BACH_PLOS_ONE/BACH_Split'},
        'NCT-CRC':  {'loader': 'nct',      'data_dir': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Split',
                     'split_file': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Results_Final/nct_crc_subset_splits.csv'},
    },
    'run': ['BreakHis', 'BACH', 'NCT-CRC'],   # remove a name to skip that dataset
}
SEED = CFG['seed']
OUT = CFG['out_dir']
for sub in ['', 'models', 'predictions', 'fold_results', 'histories', 'tables']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
json.dump({k: v for k, v in CFG.items()}, open(f'{OUT}/config_used.json', 'w'), indent=2, default=str)
print('Outputs ->', OUT)

Outputs -> /content/drive/MyDrive/R25_Control


## 1. Helpers (copied from the original notebooks)

In [3]:
def retry(fn, *a, tries=5, wait=3, **k):
    for i in range(tries):
        try:
            return fn(*a, **k)
        except OSError as e:
            if i == tries - 1:
                raise
            print(f'   ⚠️ {e} — retrying in {wait}s'); time.sleep(wait)

def listdir(p):
    return sorted(retry(os.listdir, p)) if os.path.isdir(p) else []

# ---- get_patient_data(): BreakHis version ----
def load_breakhis(data_dir, cr_level):
    rows = []
    for split in ['train', 'val', 'test']:
        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(data_dir, cr_level, split, class_name)
            for mag in listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue
                for fn in listdir(mag_dir):
                    if fn.endswith('.png'):
                        parts = fn.replace('.png', '').split('-')
                        pid = f'{parts[1]}-{parts[2]}' if len(parts) >= 3 else 'unknown'
                        rows.append(dict(filepath=os.path.join(mag_dir, fn), filename=fn, class_name=class_name,
                                         class_idx=class_idx, patient_id=pid, magnification=mag, cr_level=cr_level))
    return pd.DataFrame(rows)

# ---- get_patient_data(): BACH version (filename = patient_id) ----
def load_bach(data_dir, cr_level):
    rows = []
    for split in ['train', 'val', 'test']:
        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(data_dir, cr_level, split, class_name)
            for fn in listdir(class_dir):
                if fn.endswith('.png'):
                    rows.append(dict(filepath=os.path.join(class_dir, fn), filename=fn, class_name=class_name,
                                     class_idx=class_idx, patient_id=fn.replace('.png', ''),
                                     magnification='N/A', cr_level=cr_level))
    return pd.DataFrame(rows)

LOADERS = {'breakhis': load_breakhis, 'bach': load_bach}

def to_local(df, data_dir):
    # copy files once to local disk; return df with filepath pointing at the local copy
    def one(src):
        dst = os.path.join(LOCAL, os.path.relpath(src, os.path.dirname(data_dir)))
        if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            retry(shutil.copyfile, src, dst)
        return dst
    srcs = df['filepath'].tolist()
    t0 = time.time()
    with ThreadPoolExecutor(16) as ex:
        dsts = list(ex.map(one, srcs))
    print(f'   local copy ready: {len(dsts):,} files ({time.time() - t0:.0f}s)')
    out = df.copy(); out['filepath'] = dsts
    return out

def read_bytes(paths, workers=16):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda p: retry(lambda: open(p, 'rb').read()), paths))

def predict_bytes(model, byte_list, batch_size=64):
    # identical preprocessing to create_dataset()/load_dataset(): decode_png -> resize 224 -> /255
    @tf.autograph.experimental.do_not_convert
    def f(b):
        img = tf.image.decode_png(b, channels=3)
        img = tf.image.resize(img, CFG['image_size'])
        return tf.cast(img, tf.float32) / 255.0
    ds = (tf.data.Dataset.from_tensor_slices(tf.constant(byte_list))
          .map(f, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE))
    return model.predict(ds, verbose=0).astype('float32').flatten()

# ---- create_dataset(): BreakHis/BACH version (full-buffer shuffle) ----
def create_dataset(filepaths, labels, batch_size, shuffle=True):
    @tf.autograph.experimental.do_not_convert
    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CFG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ---- load_dataset(): NCT version (buffer capped at 10,000) ----
def load_dataset_nct(filepaths, labels, batch_size=32, shuffle=True):
    @tf.autograph.experimental.do_not_convert
    def load_and_preprocess(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CFG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(filepaths), 10000), seed=SEED)
    return ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def nct_class_weights(labels):   # get_class_weights() from the NCT notebook
    counts = np.bincount(labels.astype(int)); total = len(labels)
    return {i: total / (len(counts) * c) for i, c in enumerate(counts)}

print('✅ helpers ready')

✅ helpers ready


In [4]:
# ============================================================
# Models — copied verbatim from the notebooks that produced the reported CAT results
# ============================================================
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomZoom(0.15),
], name='augmentation')

class ChannelAttention(layers.Layer):
    def __init__(self, ratio=4, **kwargs):
        super().__init__(**kwargs); self.ratio = ratio
    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_dense1 = layers.Dense(channel // self.ratio, activation='relu')
        self.shared_dense2 = layers.Dense(channel, activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=1, keepdims=True)
        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))
        max_out = self.shared_dense2(self.shared_dense1(max_pool))
        return x * (avg_out + max_out)
    def get_config(self):
        c = super().get_config(); c.update({'ratio': self.ratio}); return c

def build_inception_densenet_attention(input_shape=(224, 224, 3)):   # BreakHis / BACH
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    inception = InceptionV3(weights='imagenet', include_top=False, pooling='avg'); inception.trainable = False
    x1 = inception(x)
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling='avg'); densenet169.trainable = False
    x2 = densenet169(x)
    x1 = layers.Dense(512, activation='relu', name='inception_proj')(x1); x1 = layers.BatchNormalization()(x1)
    x2 = layers.Dense(512, activation='relu', name='densenet_proj')(x2); x2 = layers.BatchNormalization()(x2)
    stacked = layers.Concatenate(axis=1)([layers.Reshape((1, 512))(x1), layers.Reshape((1, 512))(x2)])
    fused = layers.Flatten()(ChannelAttention(ratio=4)(stacked))
    x = layers.BatchNormalization()(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name='Inception_DenseNet169_Attention')

def build_nct_model(input_shape=(224, 224, 3)):   # NCT-CRC (concatenation + CBAM gate)
    inputs = layers.Input(shape=input_shape)
    x = layers.RandomRotation(0.15)(inputs)
    x = layers.RandomFlip('horizontal')(x)
    x = layers.RandomFlip('vertical')(x)
    x = layers.RandomZoom(0.15)(x)
    inception_base = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape, pooling='avg')
    inception_base.trainable = False
    inception_out = inception_base(x)
    densenet_base = DenseNet169(weights='imagenet', include_top=False, input_shape=input_shape, pooling='avg')
    densenet_base.trainable = False
    densenet_out = densenet_base(x)
    inc_proj = layers.BatchNormalization()(layers.Dense(512, activation='relu')(inception_out))
    dense_proj = layers.BatchNormalization()(layers.Dense(512, activation='relu')(densenet_out))
    fused = layers.Concatenate(axis=-1)([layers.Reshape((1, 1, 512))(inc_proj), layers.Reshape((1, 1, 512))(dense_proj)])
    gap = layers.GlobalAveragePooling2D()(fused)
    gap_mlp = layers.Dense(1024)(layers.Dense(256, activation='relu')(gap))
    gmp = layers.GlobalMaxPooling2D()(fused)
    gmp_mlp = layers.Dense(1024)(layers.Dense(256, activation='relu')(gmp))
    attention = layers.Reshape((1, 1, 1024))(layers.Activation('sigmoid')(layers.Add()([gap_mlp, gmp_mlp])))
    refined = layers.Multiply()([fused, attention])
    flat = layers.BatchNormalization()(layers.Flatten()(refined))
    flat = layers.Dropout(0.5)(flat)
    fc = layers.Dropout(0.3)(layers.Dense(256, activation='relu')(flat))
    output = layers.Dense(1, activation='sigmoid')(fc)
    return Model(inputs=inputs, outputs=output)

# parameter-count check against the saved models loaded in Tier A
keras.mixed_precision.set_global_policy('float32')
m = build_inception_densenet_attention(); n1 = m.count_params(); del m
m = build_nct_model(); n2 = m.count_params(); del m
keras.backend.clear_session()
print(f'BreakHis/BACH model params: {n1:,} (saved models: 36,749,793) {"✅" if n1 == 36749793 else "❌"}')
print(f'NCT-CRC model params:       {n2:,} (saved models: 37,669,217) {"✅" if n2 == 37669217 else "❌"}')

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
51877672/51877672 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
BreakHis/BACH model params: 36,749,793 (saved models: 36,749,793) ✅
NCT-CRC model params:       37,669,217 (saved models: 37,669,217) ✅


## 2. Fold reconstruction check
Before any training, this confirms the control uses **exactly** the same test images per fold as the CAT and robustness predictions from Tier A.

In [5]:
def rotating_test_folds(image_df):
    pc = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients, labels = pc['patient_id'].values, pc['class_idx'].values
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    return [(set(patients[tr]), set(patients[te])) for tr, te in skf.split(patients, labels)]

DATA = {}
for ds in ['BreakHis', 'BACH']:
    if ds not in CFG['run']:
        continue
    c = CFG['datasets'][ds]
    dfs = {cr: LOADERS[c['loader']](c['data_dir'], cr) for cr in CFG['cr_levels']}
    control_df = pd.concat([dfs['CR1_original']] * CFG['n_copies'], ignore_index=True)
    folds = rotating_test_folds(control_df)
    tA = f"{CFG['tierA_pred_dir']}/{ds}_test_folds.json"
    if os.path.exists(tA):
        ref = json.load(open(tA))
        same = all(sorted(map(str, folds[k][1])) == ref[f'fold{k+1}'] for k in range(CFG['n_folds']))
        print(f'{ds}: {len(dfs["CR1_original"]):,} CR1 images, control set {len(control_df):,} | '
              f'test folds identical to Tier A / CAT: {"✅" if same else "❌ STOP — send me this output"}')
    else:
        print(f'{ds}: Tier A fold file not found ({tA}) — cannot verify')
    DATA[ds] = dict(dfs=dfs, folds=folds)

if 'NCT-CRC' in CFG['run']:
    c = CFG['datasets']['NCT-CRC']
    sampled = pd.read_csv(c['split_file'])   # saved in the original `sampled` row order
    print(f"NCT-CRC: split file rows {len(sampled):,} | {sampled['split'].value_counts().to_dict()}")
    def nct_paths(df, cr):
        return [os.path.join(c['data_dir'], cr, s, k, f) for s, k, f in
                zip(df['original_split'], df['original_class_folder'], df['filename'])]
    DATA['NCT-CRC'] = dict(sampled=sampled, nct_paths=nct_paths)

BreakHis: 7,910 CR1 images, control set 39,550 | test folds identical to Tier A / CAT: ✅
BACH: 400 CR1 images, control set 2,000 | test folds identical to Tier A / CAT: ✅
NCT-CRC: split file rows 7,799 | {'train': 5462, 'test': 1170, 'val': 1167}


## 3. Train the CR1×5 control
Each fold: train → save the model → predict on the test images at **all 7** compression levels → save per-image predictions. A fold whose prediction file exists is skipped, so after a disconnect re-run Sections 0–2 and this cell.

In [6]:
def ram_gb():
    try:
        import psutil; return psutil.Process(os.getpid()).memory_info().rss / 1e9
    except Exception:
        return float('nan')

def fold_done(ds, k):
    return os.path.exists(f'{OUT}/predictions/{ds}_fold{k}.csv.gz')

def save_fold(ds, k, rows, fold_info, history):
    pd.concat(rows).to_csv(f'{OUT}/predictions/{ds}_fold{k}.csv.gz', index=False, compression='gzip')
    pd.DataFrame([fold_info]).to_csv(f'{OUT}/fold_results/{ds}_fold{k}.csv', index=False)
    json.dump({kk: [float(v) for v in vv] for kk, vv in history.history.items()},
              open(f'{OUT}/histories/{ds}_fold{k}.json', 'w'))

def run_rotating(ds):
    keras.mixed_precision.set_global_policy('float32')
    c = CFG['datasets'][ds]; dfs, folds = DATA[ds]['dfs'], DATA[ds]['folds']
    cr1_local = to_local(dfs['CR1_original'], c['data_dir'])
    control_df = pd.concat([cr1_local] * CFG['n_copies'], ignore_index=True)
    for k in range(1, CFG['n_folds'] + 1):
        if fold_done(ds, k):
            print(f'  {ds} fold {k}: done, skipping'); continue
        train_patients, test_patients = folds[k - 1]
        # 85/15 train/val split of training patients (original code shuffled list(set) -> session-dependent order;
        # sorted() makes it reproducible)
        tp = sorted(train_patients); np.random.seed(SEED + k - 1); np.random.shuffle(tp)
        n_train = int(len(tp) * 0.85)
        val_p, tr_p = set(tp[n_train:]), set(tp[:n_train])
        train_df = control_df[control_df.patient_id.isin(tr_p)]
        val_df = control_df[control_df.patient_id.isin(val_p)]
        cw = compute_class_weight('balanced', classes=np.unique(train_df.class_idx), y=train_df.class_idx.values)
        tf.keras.utils.set_random_seed(SEED)
        model = build_inception_densenet_attention()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=CFG['learning_rate']),
                      loss='binary_crossentropy', metrics=['accuracy'])
        cbs = [EarlyStopping(monitor='val_loss', patience=CFG['early_stopping_patience'], restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CFG['reduce_lr_patience'])]
        print(f'\n  {ds} fold {k}: train {len(train_df):,} ({len(tr_p)} patients) | val {len(val_df):,} '
              f'({len(val_p)}) | test patients {len(test_patients)}')
        t0 = time.time()
        hist = model.fit(create_dataset(train_df.filepath.values, train_df.class_idx.values, CFG['batch_size'], True),
                         validation_data=create_dataset(val_df.filepath.values, val_df.class_idx.values, CFG['batch_size'], False),
                         epochs=CFG['epochs'], class_weight={0: cw[0], 1: cw[1]}, callbacks=cbs, verbose=2)
        train_s = time.time() - t0
        model.save(f'{OUT}/models/{ds}_CR1x5_fold{k}.keras')
        rows = []
        for cr in CFG['cr_levels']:
            te = dfs[cr][dfs[cr].patient_id.isin(test_patients)]
            probs = predict_bytes(model, read_bytes(te.filepath.tolist()))
            rows.append(pd.DataFrame(dict(dataset=ds, exp='cr1x5', fold=k, train_cr='CR1x5', test_cr=cr,
                                          filename=te.filename.values, patient_id=te.patient_id.values,
                                          magnification=te.magnification.values, y_true=te.class_idx.values, prob=probs)))
            acc = ((probs >= .5).astype(int) == te.class_idx.values).mean()
            print(f'     test {cr:13s} acc={acc:.4f}')
        save_fold(ds, k, rows, dict(dataset=ds, fold=k, n_train=len(train_df), n_val=len(val_df),
                                    epochs_trained=len(hist.history['loss']), train_seconds=round(train_s)), hist)
        del model; gc.collect(); keras.backend.clear_session()
        print(f'  {ds} fold {k}: saved ({train_s/60:.1f} min training) | process RAM {ram_gb():.1f} GB')

def run_nct():
    keras.mixed_precision.set_global_policy('mixed_float16')   # as in the NCT notebook
    d = DATA['NCT-CRC']; sampled, nct_paths = d['sampled'], d['nct_paths']
    c = CFG['datasets']['NCT-CRC']
    tv = sampled[sampled.split.isin(['train', 'val'])].reset_index(drop=True)
    te = sampled[sampled.split == 'test'].reset_index(drop=True)
    tv_src = pd.DataFrame({'filepath': nct_paths(tv, 'CR1_original')})
    tv_local = to_local(tv_src, c['data_dir'])['filepath'].values
    # CAT: concat over 5 CR levels of the same train/val rows -> identical label sequence here
    paths = np.concatenate([tv_local] * CFG['n_copies']); labels = np.concatenate([tv['label'].values] * CFG['n_copies'])
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    test_bytes = {cr: read_bytes(nct_paths(te, cr)) for cr in CFG['cr_levels']}
    for k, (tr_idx, vl_idx) in enumerate(skf.split(paths, labels), start=1):
        if fold_done('NCT-CRC', k):
            print(f'  NCT-CRC fold {k}: done, skipping'); continue
        tf.keras.utils.set_random_seed(SEED)
        model = build_nct_model()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=CFG['learning_rate']),
                      loss='binary_crossentropy', metrics=['accuracy'])
        cbs = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)]
        print(f'\n  NCT-CRC fold {k}: train {len(tr_idx):,} | val {len(vl_idx):,} | test {len(te):,}')
        t0 = time.time()
        hist = model.fit(load_dataset_nct(paths[tr_idx], labels[tr_idx], CFG['batch_size'], True),
                         validation_data=load_dataset_nct(paths[vl_idx], labels[vl_idx], CFG['batch_size'], False),
                         epochs=CFG['epochs'], callbacks=cbs, class_weight=nct_class_weights(labels[tr_idx]), verbose=2)
        train_s = time.time() - t0
        model.save(f'{OUT}/models/NCT-CRC_CR1x5_fold{k}.keras')
        rows = []
        for cr in CFG['cr_levels']:
            probs = predict_bytes(model, test_bytes[cr])
            rows.append(pd.DataFrame(dict(dataset='NCT-CRC', exp='cr1x5', fold=k, train_cr='CR1x5', test_cr=cr,
                                          filename=te.filename.values, patient_id=te.filename.str.rsplit('.', n=1).str[0].values,
                                          magnification='N/A', y_true=te.label.values, prob=probs)))
            print(f'     test {cr:13s} acc={((probs >= .5).astype(int) == te.label.values).mean():.4f}')
        save_fold('NCT-CRC', k, rows, dict(dataset='NCT-CRC', fold=k, n_train=len(tr_idx), n_val=len(vl_idx),
                                           epochs_trained=len(hist.history['loss']), train_seconds=round(train_s)), hist)
        del model; gc.collect(); keras.backend.clear_session()
        print(f'  NCT-CRC fold {k}: saved ({train_s/60:.1f} min training) | process RAM {ram_gb():.1f} GB')
    keras.mixed_precision.set_global_policy('float32')

T0 = time.time()
for ds in CFG['run']:
    print(f'\n{"=" * 60}\n  {ds}: CR1×5 control\n{"=" * 60}')
    run_nct() if ds == 'NCT-CRC' else run_rotating(ds)
print(f'\n✅ training finished in {(time.time() - T0)/3600:.2f} h')


  BreakHis: CR1×5 control
   local copy ready: 7,910 files (372s)
  BreakHis fold 1: done, skipping
  BreakHis fold 2: done, skipping
  BreakHis fold 3: done, skipping
  BreakHis fold 4: done, skipping
  BreakHis fold 5: done, skipping

  BACH: CR1×5 control
   local copy ready: 400 files (24s)
  BACH fold 1: done, skipping
  BACH fold 2: done, skipping
  BACH fold 3: done, skipping
  BACH fold 4: done, skipping
  BACH fold 5: done, skipping

  NCT-CRC: CR1×5 control
   local copy ready: 6,629 files (373s)
  NCT-CRC fold 1: done, skipping
  NCT-CRC fold 2: done, skipping

  NCT-CRC fold 3: train 26,516 | val 6,629 | test 1,170
Epoch 1/30
829/829 - 162s - 195ms/step - accuracy: 0.9568 - loss: 0.1177 - val_accuracy: 0.9846 - val_loss: 0.0436 - learning_rate: 0.0010
Epoch 2/30
829/829 - 113s - 137ms/step - accuracy: 0.9749 - loss: 0.0673 - val_accuracy: 0.9820 - val_loss: 0.0450 - learning_rate: 0.0010
Epoch 3/30
829/829 - 114s - 137ms/step - accuracy: 0.9786 - loss: 0.0576 - val_accurac

## 4. Analysis: volume effect vs diversity effect (answer to R2-5)
* **Robustness** = 1× CR1 training (Tier A predictions)
* **CR1×5 control** = 5× CR1 training (this notebook)
* **CAT** = CR1 + CR15 + CR25 + CR50 + CR100 (Tier A predictions)

Volume effect = control − robustness. Diversity effect = CAT − control. Same test images, paired bootstrap (by patient for BreakHis, by image for BACH and NCT-CRC), 10,000 resamples, permutation p-values, Holm correction within each dataset.

In [7]:
def load(ds, src):
    d = CFG['tierA_pred_dir'] if src == 'tierA' else f'{OUT}/predictions'
    fs = sorted(glob.glob(f'{d}/{ds}_fold*.csv.gz'))
    return pd.concat([pd.read_csv(f) for f in fs]) if fs else None

rng = np.random.default_rng(SEED); B = CFG['n_boot']
FIXED = {'NCT-CRC'}

def paired(P, ds, a, b):
    ka = P[(P.exp == a[0]) & (P.test_cr == a[1])]; kb = P[(P.exp == b[0]) & (P.test_cr == b[1])]
    key = ['filename'] if ds in FIXED else ['fold', 'filename']
    j = ka.merge(kb, on=key, suffixes=('_a', '_b'))
    j['ca'] = ((j.prob_a >= .5).astype(int) == j.y_true_a).astype(float)
    j['cb'] = ((j.prob_b >= .5).astype(int) == j.y_true_b).astype(float)
    if ds in FIXED:
        j = j.groupby('filename').agg(ca=('ca', 'mean'), cb=('cb', 'mean'), y=('y_true_a', 'first'),
                                      cluster=('patient_id_a', 'first')).reset_index()
    else:
        j = j.rename(columns={'y_true_a': 'y', 'patient_id_a': 'cluster'})
    return j

def boot(j):
    j = j.assign(ca_pos=j.ca * (j.y == 1), cb_pos=j.cb * (j.y == 1), pos=(j.y == 1).astype(float))
    cl = j.groupby('cluster').agg(sa=('ca', 'sum'), sb=('cb', 'sum'), n=('ca', 'size'),
                                  pa=('ca_pos', 'sum'), pb=('cb_pos', 'sum'), npos=('pos', 'sum'))
    sa, sb, n, pa, pb, npos = [cl[c].values for c in ['sa', 'sb', 'n', 'pa', 'pb', 'npos']]
    na_, nb_, nneg = sa - pa, sb - pb, n - npos
    delta = (sa.sum() - sb.sum()) / n.sum()
    bal = lambda p_, n_, w: 0.5 * ((w @ p_) / (w @ npos) + (w @ n_) / (w @ nneg))
    one = np.ones(len(n))
    W = rng.multinomial(len(n), np.full(len(n), 1 / len(n)), size=B).astype(float)
    db = (W @ (sa - sb)) / (W @ n)
    with np.errstate(invalid='ignore', divide='ignore'):
        dbal_b = 0.5 * ((W @ pa) / (W @ npos) + (W @ na_) / (W @ nneg)) - 0.5 * ((W @ pb) / (W @ npos) + (W @ nb_) / (W @ nneg))
    S = rng.choice([-1.0, 1.0], size=(B, len(n)))
    p = (1 + np.sum(np.abs((S @ (sa - sb)) / n.sum()) >= abs(delta) - 1e-12)) / (B + 1)
    return dict(delta_acc_pp=100 * delta, ci_low=100 * np.percentile(db, 2.5), ci_high=100 * np.percentile(db, 97.5),
                delta_bal_acc_pp=100 * (bal(pa, na_, one) - bal(pb, nb_, one)),
                bal_ci_low=100 * np.nanpercentile(dbal_b, 2.5), bal_ci_high=100 * np.nanpercentile(dbal_b, 97.5),
                p_perm=p, n_images=int(n.sum()), n_clusters=len(n))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, idx in enumerate(o):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(1, run)
    return adj

acc_rows, cmp_rows = [], []
for ds in CFG['run']:
    A, Ctl = load(ds, 'tierA'), load(ds, 'control')
    if A is None or Ctl is None:
        print(f'{ds}: missing predictions (Tier A: {A is not None}, control: {Ctl is not None})'); continue
    P = pd.concat([A[A.exp.isin(['robustness', 'cat'])], Ctl], ignore_index=True)
    P['correct'] = ((P.prob >= .5).astype(int) == P.y_true).astype(float)
    fa = P.groupby(['exp', 'test_cr', 'fold'])['correct'].mean().reset_index()
    for cr in CFG['cr_levels']:
        r = dict(dataset=ds, test_cr=cr)
        for e, lab in [('robustness', 'robustness_1x'), ('cr1x5', 'control_CR1x5'), ('cat', 'CAT')]:
            v = fa[(fa.exp == e) & (fa.test_cr == cr)]['correct'] * 100
            r[f'{lab}_mean'], r[f'{lab}_sd'] = v.mean(), v.std()
        acc_rows.append(r)
        for fam, a, b in [('volume (control − robustness)', ('cr1x5', cr), ('robustness', cr)),
                          ('diversity (CAT − control)', ('cat', cr), ('cr1x5', cr))]:
            cmp_rows.append(dict(dataset=ds, effect=fam, test_cr=cr, **boot(paired(P, ds, a, b))))

ACC = pd.DataFrame(acc_rows).round(2)
CMP = pd.DataFrame(cmp_rows)
CMP['p_holm'] = np.nan
for ds in CMP.dataset.unique():
    m = CMP.dataset == ds; CMP.loc[m, 'p_holm'] = holm(CMP.loc[m, 'p_perm'].values)
CMP = CMP.round(4)
ACC.to_csv(f'{OUT}/tables/r25_accuracy_by_condition.csv', index=False)
CMP.to_csv(f'{OUT}/tables/r25_volume_vs_diversity.csv', index=False)
pd.set_option('display.width', 250)
print('Accuracy (%), mean ± SD over 5 folds'); print(ACC.to_string(index=False))
print('\nEffects (pp) with 95% bootstrap CI and Holm-adjusted p'); print(CMP.to_string(index=False))

Accuracy (%), mean ± SD over 5 folds
 dataset      test_cr  robustness_1x_mean  robustness_1x_sd  control_CR1x5_mean  control_CR1x5_sd  CAT_mean  CAT_sd
BreakHis CR1_original               82.79              6.50               84.70              4.01     86.30    5.81
BreakHis         CR15               82.48              6.48               84.40              4.14     86.35    5.72
BreakHis         CR25               82.35              6.52               84.06              4.19     86.34    5.75
BreakHis         CR50               82.39              6.61               83.95              4.46     86.15    6.11
BreakHis        CR100               80.97              7.14               82.58              4.75     85.99    5.68
BreakHis        CR500               70.10             10.95               73.68              6.56     78.96    7.43
BreakHis       CR1000               61.48             14.60               67.48              8.58     74.89    8.40
    BACH CR1_original              

In [8]:
# ---- one-line verdict per dataset, and the per-fold training record ----
for ds in CMP.dataset.unique():
    d = CMP[CMP.dataset == ds]
    print(f'\n{ds}:')
    for cr in CFG['cr_levels']:
        v = d[(d.test_cr == cr) & d.effect.str.startswith('volume')].iloc[0]
        g = d[(d.test_cr == cr) & d.effect.str.startswith('diversity')].iloc[0]
        print(f'  {cr:13s} volume {v.delta_acc_pp:+6.2f} [{v.ci_low:+.2f}, {v.ci_high:+.2f}] p_holm={v.p_holm:.3f} | '
              f'diversity {g.delta_acc_pp:+6.2f} [{g.ci_low:+.2f}, {g.ci_high:+.2f}] p_holm={g.p_holm:.3f}')
FR = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/fold_results/*.csv'))])
print('\nTraining record:'); print(FR.to_string(index=False))
FR.to_csv(f'{OUT}/tables/r25_training_record.csv', index=False)
print(f'\nSend me: {OUT}/tables/ (3 CSVs). Predictions and models stay in {OUT} for the repository (R1-8).')


BreakHis:
  CR1_original  volume  +1.87 [-1.04, +4.85] p_holm=1.000 | diversity  +1.44 [-0.72, +3.50] p_holm=1.000
  CR15          volume  +1.87 [-1.07, +4.73] p_holm=1.000 | diversity  +1.78 [-0.40, +3.90] p_holm=1.000
  CR25          volume  +1.64 [-1.27, +4.54] p_holm=1.000 | diversity  +2.11 [-0.13, +4.39] p_holm=0.813
  CR50          volume  +1.53 [-1.40, +4.41] p_holm=1.000 | diversity  +1.98 [-0.42, +4.42] p_holm=1.000
  CR100         volume  +1.53 [-1.60, +4.53] p_holm=1.000 | diversity  +3.17 [+0.37, +6.11] p_holm=0.486
  CR500         volume  +3.11 [-2.28, +8.46] p_holm=1.000 | diversity  +5.03 [+0.23, +9.93] p_holm=0.632
  CR1000        volume  +5.15 [-1.27, +11.63] p_holm=1.000 | diversity  +7.18 [+0.45, +14.25] p_holm=0.615

BACH:
  CR1_original  volume  +4.25 [-0.25, +9.00] p_holm=1.000 | diversity  -0.25 [-4.25, +3.75] p_holm=1.000
  CR15          volume  +3.75 [-0.75, +8.25] p_holm=1.000 | diversity  +0.00 [-3.75, +3.75] p_holm=1.000
  CR25          volume  +4.00 [-0.2